# build_affected_range
`stg_ghsa_normalized`(`ghsa_id` + `vulnerabilities_raw`) → `silver_advisory_ghsa_affected`.
Grain: `ghsa_id` × `package_key` × `range`. 1 GHSA × N개의 영향 패키지 = N행 (explode).

**Business Key는 `range_idx`가 아니라 `ghsa_id + package_key + vulnerable_version_range`.**
`range_idx`(원본 배열 순번)는 lineage 추적용으로만 남긴다 — GHSA API가 배열 순서를 보장하지 않으면 내용은 그대로인데 키만 바뀐 것처럼 보이는 문제가 생기기 때문이다(03_Gold_Schema.md §3.7 키 설계 수정 참고, 동일 원리를 Silver 단계에도 미리 적용).

이 테이블은 valid_from/valid_to 이력을 두지 않는다(§3.10) — **오늘 변경된 GHSA만** 골라 그 GHSA의 기존 행을 지우고 새로 넣는 "GHSA 단위 전체 교체" 방식이라, GHSA당 항상 최신 상태 하나만 남는다.

In [ ]:
%run 00_silver_common

In [ ]:
norm_df = spark.table(TBL["stg_ghsa_normalized"]).select(
    "ghsa_id", "vulnerabilities_raw", "batch_id", "ingested_at"
)

exploded = (
    norm_df
    .select(
        "ghsa_id", "batch_id", "ingested_at",
        F.posexplode_outer(F.col("vulnerabilities_raw")).alias("range_idx", "vuln"),
    )
)

vuln_cols = exploded.schema["vuln"].dataType.fieldNames() if "vuln" in exploded.columns else []

def vuln_field(name, default=None):
    return F.col(f"vuln.{name}") if name in vuln_cols else F.lit(default)

package_struct_cols = []
if "package" in vuln_cols:
    package_struct_cols = exploded.schema["vuln"].dataType["package"].dataType.fieldNames()

def package_field(name, default=None):
    if "package" in vuln_cols and name in package_struct_cols:
        return F.col("vuln.package")[name]
    return F.lit(default)


In [ ]:
from pyspark.sql.types import StructType

ecosystem = package_field("ecosystem")
package_name = package_field("name")
vulnerable_version_range = vuln_field("vulnerable_version_range")

# first_patched_version: 문자열이거나 {identifier: ...} 객체일 수 있다.
if "first_patched_version" in vuln_cols:
    fpv_type = exploded.schema["vuln"].dataType["first_patched_version"].dataType
    if isinstance(fpv_type, StructType) and "identifier" in fpv_type.fieldNames():
        first_patched_version = F.col("vuln.first_patched_version.identifier")
    else:
        first_patched_version = F.col("vuln.first_patched_version")
else:
    first_patched_version = F.lit(None)

vulnerable_functions = vuln_field("vulnerable_functions")
vulnerable_functions = F.coalesce(vulnerable_functions, F.array().cast("array<string>"))

package_key = F.concat(F.lower(ecosystem), F.lit(":"), F.lower(package_name))


In [ ]:
dq_flags = flags_array(
    F.when(ecosystem.isNull() | package_name.isNull(), F.lit("MISSING_PACKAGE_INFO")),
    F.when(vulnerable_version_range.isNull(), F.lit("MISSING_VERSION_RANGE")),
    F.when(first_patched_version.isNull(), F.lit("NO_PATCH_AVAILABLE")),
)

affected_df = exploded.select(
    "ghsa_id",
    ecosystem.alias("ecosystem"),
    package_name.alias("package_name"),
    vulnerable_version_range.alias("vulnerable_version_range"),
    first_patched_version.alias("first_patched_version"),
    vulnerable_functions.alias("vulnerable_functions"),
    package_key.alias("package_key"),
    "range_idx",
    dq_flags.alias("dq_flags"),
    F.lit("ghsa").alias("source"),
    "batch_id", "ingested_at",
).filter(F.col("ghsa_id").isNotNull() & F.col("package_key").isNotNull())

CONTENT_COLS = ["ghsa_id", "ecosystem", "package_name", "vulnerable_version_range",
                 "first_patched_version", "vulnerable_functions", "package_key",
                 "range_idx", "dq_flags", "source"]
affected_df = affected_df.withColumn("record_hash", content_hash([F.col(c) for c in CONTENT_COLS]))


In [ ]:
# GHSA에 vulnerabilities/affected 필드가 아예 없어 explode 결과가 0행인 경우(시나리오①)는
# 로그만 남기고 임의로 빈 행을 만들지 않는다 -- affected_df는 자연스럽게 그 ghsa_id를 포함하지 않는다.
no_package_info_ghsa = (
    norm_df.select("ghsa_id")
    .join(affected_df.select("ghsa_id").distinct(), on="ghsa_id", how="left_anti")
    .count()
)

target_ghsa_ids = [r["ghsa_id"] for r in norm_df.select("ghsa_id").distinct().collect()]

if table_exists(TBL["advisory_ghsa_affected"]):
    if target_ghsa_ids:
        t = DeltaTable.forName(spark, TBL["advisory_ghsa_affected"])
        t.delete(F.col("ghsa_id").isin(target_ghsa_ids))
    write_append(affected_df, TBL["advisory_ghsa_affected"])
else:
    write_overwrite(affected_df, TBL["advisory_ghsa_affected"])

affected_count = affected_df.count()
log_batch_run("build_affected_range", "SUCCESS", record_count=affected_count)
log_dq_metric("build_affected_range", "ghsa_without_package_info", no_package_info_ghsa, passed=True,
              detail="vulnerabilities/affected 필드가 없어 행이 생성되지 않은 GHSA 수")
print(f"[build_affected_range] affected_rows={affected_count} "
      f"ghsa_without_package_info={no_package_info_ghsa} batch_id={batch_id}")
